# Evaluation

Loads the trained models from `models/`, reports final train/test accuracy
for the dropout vs no-dropout nets and produces the test-set confusion
matrix for the dropout model.


# Setup (run this first)

1. Edit `REPO_URL` to point at your GitHub repo.
2. Run the cell. It clones the repo into `/content/my-research`, installs
   requirements and adds the repo to `sys.path` so `import src.*` works.

If you do not want to use GitHub yet, upload the `my-research` folder to
Google Drive and instead run:
   from google.colab import drive; drive.mount('/content/drive')
   %cd '/content/drive/MyDrive/my-research'


In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/my-research"  # <-- edit me
%cd /content
!test -d my-research || git clone --depth 1 {REPO_URL} my-research
%cd /content/my-research
!pip install -q -r requirements.txt
import os, sys
sys.path.insert(0, os.getcwd())
print('repo ready at', os.getcwd())


In [ ]:
import torch
from src.dataset import mnist_loaders, device
from src.model import Net
from src.utils import evaluate, plot_confusion_matrix

device_ = device()
train_loader, test_loader, _ = mnist_loaders(batch_size=128)

def load_model(path, use_dropout):
    model = Net(dropout=use_dropout).to(device_)
    model.load_state_dict(torch.load(path, map_location=device_))
    return model

nodrop = load_model('models/mnist_nodropout.pt', use_dropout=False)
drop = load_model('models/mnist_dropout.pt', use_dropout=True)
print('models loaded')


In [ ]:
# final numbers on the 10,000-image test set
results = {}
for name, model in [('No dropout', nodrop), ('Dropout', drop)]:
    test_acc, _ = evaluate(model, test_loader, device_)
    results[name] = test_acc
    print(f'{name:12s} | test acc {test_acc:6.3f}% | test error {100 - test_acc:5.3f}%')

best = max(results, key=results.get)
print(f"\nBest model on the test set: {best}")


In [ ]:
# confusion matrix for the dropout model (saved to results/confusion_matrix.png)
drop_acc, cm = evaluate(drop, test_loader, device_)
plot_confusion_matrix(cm)
print('at top-1 accuracy:', drop_acc)


## Compare with the paper

Srivastava et al. (2014) report for feed-forward MNIST nets:

| Model | Test error |
|---|---|
| Standard 2-layer net, 800 logistic units (Simard et al., 2003) | 1.60% |
| Dropout net, 3 layers, 1024 logistic units | 1.35% |
| Dropout net, 3 layers, 1024 ReLU units | 1.25% |
| Dropout net + max-norm, 3 layers, 1024 ReLU units | 1.06% |

Our basic architecture (2x1024 ReLU) mirrors the *method*; exact numbers
depend on hyper-parameters, epochs and network width.
